In [2]:
import torch
import torchvision.transforms as transforms

train_transforms=transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(  mean=(0.485,0.456,0.406),
      std=(0.229,0.224,0.225))
])

In [3]:
# data importer

import kagglehub

# Download latest version
path = kagglehub.dataset_download("karakaggle/kaggle-cat-vs-dog-dataset")

# import os
# print(os.listdir("/kaggle/input//kaggle-cat-vs-dog-dataset/kagglecatsanddogs_3367a/PetImages"))

100%|██████████| 787M/787M [00:09<00:00, 85.4MB/s]

Extracting files...


In [4]:
# data loader
from torchvision.datasets import ImageFolder
import os


dataset=ImageFolder(os.path.join(path, "kagglecatsanddogs_3367a/PetImages"),transform=train_transforms)


FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input//kaggle-cat-vs-dog-dataset/kagglecatsanddogs_3367a/PetImages'

In [ ]:
# split data
from torch.utils.data import random_split

data_size=len(dataset)

train_data=int(0.8*data_size)
vali_data=int(10*data_size)
test_data=data_size-train_data-vali_data

train_data,vali_data,test_data=random_split(
          dataset,

          [train_data,vali_data,test_data]

                                          )



In [ ]:
# data loader
from torch.utils.data import DataLoader

train_loader=DataLoader(train_data,batch_size=32,shuffle=True)
vali_loader=DataLoader(vali_data,batch_size=32)
test_loader=DataLoader(test_data,batch_size=32)
len(train_loader)


In [ ]:
import torchvision.models as models

import torch.nn as nn
import torch.optim as optim

model=models.resnet18(pretrained=True)

device=torch.device("cuda" if torch.cuda.is_available() else "cpu")

model=model.to(device)

criterion=nn.CrossEntropyLoss()
optimizer=optim.Adam(model.parameters(),lr=0.0001)

for i in range(10):
      for images,labels in train_loader:
         images=images.to(device)
         labels=labels.to(device)

         optimizer.zero_grad()

         output=model(images)

         loss=criterion(output,labels)

         loss.backward()

         optimizer.step()

      print("loss is :",loss)



In [ ]:
# validation fase
model.eval()
with torch.no_grad():
  correct=0
  total=0
  for images,labels in vali_loader:
       image=images.to(device)
       label=labels.to(device)

       output=model(image)

       _,predict=torch.max(output,1)

       total += label.size(0)

       correct += (predict==label).sum().item()
       print(f"validation accuracy :",100*(correct/total))







In [ ]:
# test data fase
from PIL import Image
image=Image.open("/content/pexels-sart-face-209620-672244.jpg")

image=train_transforms(image)
image=image.unsqueeze(0)

image=image.to(device)

with torch.no_grad():
  output=model(image)

_,predicted=torch.max(output,1)

print(f"so upload animal is {dataset.classes[predicted.item()]}")


In [ ]:
torch.save(model.state_dict(), "model_1.pth")
print("Model saved successfully to /content/model_1.pth")